# 01 · Limpieza: raw → interim → processed → input.json

Ejecuta el mismo pipeline que `make data` y `make json`, y muestra las validaciones. Si una validación falla, se corrige en `src/optimum/cleaning.py` (con test), nunca a mano en los datos.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from optimum import paths

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
from optimum.cleaning import build_processed, save_processed
from optimum.io.excel import export_interim

export_interim()
tables = build_processed()  # lanza DataValidationError si algún control falla
save_processed(tables)
pd.Series({k: v.shape for k, v in tables.items()}, name="shape")

## Controles

`build_processed` ya exige: fechas de fin de mes únicas, tasas en decimales, FX y equity positivos, matrices simétricas con el orden de `FACTORS`, cupón + principal = total, principal al vencimiento = nocional y probabilidad de estrés = 5 %. Aquí se muestran los resultados.

### Cambios de factores: Excel vs. derivación propia

In [ ]:
from optimum.cleaning import compute_factor_changes

ours = compute_factor_changes(tables["market_history"])
err = (tables["factor_changes"] - ours).abs().max()
print(f"Máxima diferencia: {err.max():.2e} ({len(ours)} meses x {ours.shape[1]} factores)")

### Balance inicial (S/ mm)

In [ ]:
inst = tables["instruments"]
by_side = inst.groupby("side")["market_value_pen"].sum()
by_side["PN"] = by_side["ASSET"] - by_side["LIABILITY"]
by_side

In [ ]:
inst.pivot_table(index="instrument_type", columns="currency", values="market_value_pen", aggfunc="sum", fill_value=0)

### Flujos de caja

In [ ]:
tables["cash_flows"].groupby("instrument_id").agg(
    tipo=("coupon_type", "first"),
    n=("period", "size"),
    ultimo=("payment_date", "max"),
    principal=("principal_cash_flow", "sum"),
)

### Escenarios de estrés (Caso C) y árbol del Caso D

In [ ]:
stress = tables["stress_scenarios_c"]
print(f"Probabilidad conjunta de estrés: {stress['probability'].sum():.4f}")
stress.drop(columns="note")

In [ ]:
tables["scenario_tree_d_transitions"], tables["scenario_tree_d_shocks"].shape

## Contrato: input.json

Se construye desde `config/caso.yaml` + `processed/` y se valida contra el Anexo 1.

In [ ]:
from optimum.io.json_contract import build_input, validate_input, write_json

doc = build_input()
errors = validate_input(doc)
assert not errors, errors
write_json(doc, paths.INPUT_JSON)
print(f"input.json válido: caso {doc['caseId']}, {len(doc['instruments'])} instrumentos, "
      f"{len(doc['cashFlows'])} flujos, {len(doc['marketHistory'])} fechas")